# Kalibratie — trainingen scoren op herschrijfbaarheid

Draai de scorer op een handvol trainingen, bekijk de output, en leg naast je handmatige labels.

**Voor je begint:**
1. `pip install -r requirements.txt`
2. Zet je API-key in een `.env`-bestand (zie `.env.example`).
3. Download de Google Sheet als **xlsx** (Bestand → Downloaden) en zet het pad hieronder.
4. Zet `Scoringsrubric_herschrijfbaarheid_v1.md` in dezelfde map.

In [5]:
from dotenv import load_dotenv
load_dotenv()

import importlib, score_trainings as st
importlib.reload(st)   # zodat je edits in score_trainings.py meteen meepakt

IN_XLSX   = "/Users/hugovandenbelt/Downloads/TrainingenLijst_50.xlsx"      # <- gedownloade sheet
RUBRIC    = "Scoringsrubric_herschrijfbaarheid_v1.md"
N         = 46                                        # eerst een klein setje
WEB_SEARCH = False                                      # actualiteitscheck aan

## 1. Sheet inlezen en de content-parser controleren
Check eerst dat de kolommen goed herkend worden en dat de brontekst schoon uit de JSON komt.

In [7]:
df, cols = st.read_input(IN_XLSX)
print("Herkende kolommen:", cols)
print("Aantal rijen:", len(df))

# Inspecteer de eerste training: parsed content + geschoonde brontekst
row0 = df.iloc[0]
content0 = st.parse_content(row0[cols['content']])
print("\nJSON-keys:", list(content0))
print("Dagen:", st.extract_days(content0, row0[cols['days']] if cols['days'] else None))
print("\n--- brontekst (eerste 800 tekens) ---")
print(st.build_source_text(content0, str(row0[cols['name']]))[:800])

Herkende kolommen: {'id': 'id', 'name': 'name', 'content': 'content', 'days': None}
Aantal rijen: 50

JSON-keys: ['days', 'intro', 'setup', 'modules', 'summary', 'follow_up', 'objectives', 'certification', 'summary_edudex', 'prior_knowledge', 'target_audience']
Dagen: None

--- brontekst (eerste 800 tekens) ---
[days]
5

[intro]
PHP is een van de belangrijke programmeertalen in de markt. PHP wordt veelal gebruikt om webapplicaties te ontwikkelen. Bekende software zoals Wordpress en Magento is ontwikkeld in PHP.

Tijdens de PHP Cursus

 Tijdens de Opleiding PHP Professional leren wij je uitgebreid programmeren in PHP. Je leert de belangrijkste constructies uit de taal kennen en gaat een database opzetten en implementeren in MySQL. We starten met functioneel programmeren en gaandeweg de cursus stappen we over naar object georiënteerd programmeren. 

Resultaat van de PHP Cursus

 Aan het einde van de Opleiding PHP Professional ben je in staat zelfstandig een webapplicatie te bouwen op bas

## 2. Scoren
Kost een paar seconden per training (meer met web search). Output gaat ook naar een xlsx.

In [8]:
out = st.run_file(IN_XLSX, "trainingen_scored_websearch2.xlsx", RUBRIC, limit=N, use_web_search=WEB_SEARCH)
out[['titel','eindscore','verdict','actualiteit_type','actualiteit_severity','menselijke_input_nodig','scorer_confidence']]

[1/3] Opleiding PHP Professional                    -> 54 redelijk
[2/3] Opleiding VB.NET Professional                 -> 40 dun
[3/3] Opleiding Java Professional                   -> 45 redelijk

Geschreven: trainingen_scored_websearch2.xlsx  (3 rijen)


,titel,eindscore,verdict,actualiteit_type,actualiteit_severity,menselijke_input_nodig,scorer_confidence
0,Opleiding PHP Professional,54,redelijk,additief,medium,False,medium
1,Opleiding VB.NET Professional,40,dun,structureel,high,True,high
2,Opleiding Java Professional,45,redelijk,additief,high,False,medium


## 3. Naast je handmatige labels leggen
Vul je eigen labels in (id → score). Kijk of de scores correleren en of de verdict-grenzen landen waar je ze wil.
Stel daarna in `score_trainings.py` de `IMPACT`-tabel of `WEIGHTS` bij en draai opnieuw (reload-cel bovenaan).

In [17]:
import pandas as pd

# jouw handmatige labels: {training_id: score}
labels = {
    # 43: 40,   # Cursus XSL
    # 87: 70,   # Cursus LDAP
}

cmp = out[['training_id','titel','eindscore','verdict']].copy()
cmp['label'] = cmp['training_id'].map(labels)
cmp['delta'] = (cmp['eindscore'] - cmp['label']).abs()
cmp.dropna(subset=['label'])

,training_id,titel,eindscore,verdict,label,delta


In [ ]:
# Verdeling van de scores over de populatie — zie hoe scheef je input is
out['verdict'].value_counts().reindex(['al_nieuwe_stijl','rijk','redelijk','dun','onbruikbaar']).fillna(0).astype(int)

## 4. Volledige feedback van één training bekijken
Handig om te zien of de vooruitgerichte feedback (bruikbaar / strippen / gaten / rewrite_guidance) klopt.

In [8]:
import textwrap
r = out.iloc[0]
for k in ['titel','kern','eindscore','verdict','actualiteit_actie','bruikbaar','strippen','gaten','rewrite_guidance']:
    print(f"{k}:")
    print(textwrap.fill(str(r[k]), 100, initial_indent='  ', subsequent_indent='  '))
    print()

titel:
  Opleiding PHP Professional

kern:
  PHP Professional leert (junior) developers programmeren in PHP, van functioneel naar object-
  georiënteerd, met MySQL-database-implementatie, en levert een zelf gebouwde webapplicatie
  (praktijkcase webwinkel) op.

eindscore:
  58

verdict:
  redelijk

actualiteit_actie:
  refresh: voeg moderne PHP-ecosysteemonderdelen toe (Composer, PSR-4 autoloading, namespaces, PHP
  8.x taalfeatures) aan de OO-module en check of MySQL-module PDO/prepared statements benoemt in
  plaats van alleen klassieke queries.

bruikbaar:
  Resultaat-sectie met expliciete leeruitkomst (zelfstandig webapplicatie bouwen) | Praktijkcase
  webwinkel als rode draad | Vier modules met concrete sub-onderwerpen: Inleiding programmeren
  (array's, regex, cookies, sessies, exception handling, debugging), MySQL, OO programmeren
  (classes/objects, exception handling, design patterns) | Security als doorlopend thema (input-
  validatie, veilige database-toegang) | Eindopdracht